In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
import gradio as gr
from utils.util import get_llm, LLM
from utils.util import message_updater
load_dotenv(override=True)

In [ ]:
anthropic1 = get_llm(LLM.ANTROPIC)
anthropic2 = get_llm(LLM.ANTROPIC)
ANTHROPIC_MODEL = 'claude-sonnet-4-5-20250929'
system_message = "You are a chat bot which will talk with another chat bot. \
    Your responses should be not more than 2 to 3 short sentenses. Make sure you move conversation ahead."

def start_chat(topic):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': topic}
    ]
    chat_history = []

    for i in range(5):
        message_updater(messages=messages, role='user')
        response = anthropic1.chat.completions.create(model=ANTHROPIC_MODEL, messages=messages)
        bot1_content = response.choices[0].message.content
        print(f"Bot 1: {bot1_content}")
        messages.append({'role': 'assistant', 'content': bot1_content})
        chat_history.append({"role": "assistant", "content": f"**Bot 1:** {bot1_content}"})
        yield chat_history

        message_updater(messages=messages, role='assistant')
        response = anthropic2.chat.completions.create(model=ANTHROPIC_MODEL, messages=messages)
        bot2_content = response.choices[0].message.content
        print(f"Bot 2: {bot2_content}")
        messages.append({'role': 'assistant', 'content': bot2_content})
        chat_history.append({"role": "user", "content": f"**Bot 2:** {bot2_content}"})
        yield chat_history


with gr.Blocks() as demo:
    gr.Markdown("## LLM Fireside Chat")
    chatbot = gr.Chatbot(type="messages", height=500, label="Bot 1 (left) vs Bot 2 (right)")
    with gr.Row():
        topic_box = gr.Textbox(value="lets discuss about Marketcast", label="Topic", scale=4)
        start_btn = gr.Button("Start Chat", variant="primary", scale=1)

    start_btn.click(fn=start_chat, inputs=topic_box, outputs=chatbot)

demo.launch()